# 02 — Transform: 2026 half-PPR draft checkpoint

Demonstrates the first reusable transformation layer
(`fantasy_football.transform.draft_checkpoint`) against the **cached** raw
payloads in `data/raw/`. No API calls are made here.

The transformation logic lives in `src/` — this notebook only calls it,
inspects the result, runs lightweight QA, and writes the Excel checkpoint.

Pipeline (see the module docstring for detail):

1. **Spine** = 2026 half-PPR consensus ADP response (primary player universe).
2. LEFT JOIN on `player_id`: ESPN PPR ADP, 2026 projected half-PPR points,
   2025 actual half-PPR production.
3. Derive `rank_range` and `adp_vs_ecr`.
4. Export a single-sheet review workbook to `outputs/`.

In [1]:
import json
from pathlib import Path

import pandas as pd

from fantasy_football.transform import (
    CHECKPOINT_COLUMNS,
    RANK_ECR_VS_ADP_NOTE,
    build_draft_checkpoint,
    checkpoint_coverage_report,
    write_checkpoint_excel,
)

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

## Build the consolidated checkpoint dataframe

In [2]:
df = build_draft_checkpoint()  # defaults to <repo>/data/raw
print("shape:", df.shape)
df.head(15)

shape: (365, 27)


,player_name,team,position,pos_rank,bye_wk,rank_ecr,consensus_adp_half,tier,rank_min,rank_max,...,2025_games_played,2025_points_half,2025_ppg_half,projected_points_half,adp_vs_ecr,eligible_positions,player_id,sportsdata_id,player_yahoo_id,cbs_player_id
0,Jahmyr Gibbs,DET,RB,RB1,6,1,1.00,<NA>,1,1,...,17,328.4,19.3,337.38,0.00,RB,22968,fef9457e-6497-47de-9bf2-cc3b95929375,40059,3162723
1,Bijan Robinson,ATL,RB,RB2,11,2,2.00,<NA>,2,2,...,17,331.3,19.5,318.38,0.00,RB,23133,f78d68c2-f9da-48e7-b954-26b69efd828d,40055,3168163
2,Ja'Marr Chase,CIN,WR,WR1,6,3,3.00,<NA>,3,3,...,16,251.1,15.7,275.38,0.00,WR,19788,fa99e984-d63b-4ef4-a164-407f68a7eeaf,33393,2966320
3,Puka Nacua,LAR,WR,WR2,11,4,4.67,<NA>,4,6,...,16,310.5,19.4,281.30,0.67,WR,23180,111be44d-7bc2-4cad-934d-c9e946293b2f,40168,3121687
4,Christian McCaffrey,SF,RB,RB3,8,5,4.67,<NA>,4,5,...,17,365.6,21.5,295.45,-0.33,RB,16393,f96db0af-5e25-42d1-a07a-49b4e065b364,30121,2136743
5,Jonathan Taylor,IND,RB,RB4,13,6,6.00,<NA>,5,7,...,17,339.3,20.0,288.69,0.00,RB,19217,925195a4-06ba-4e37-ae7d-a3d6a5419139,32711,2866395
6,Jaxon Smith-Njigba,SEA,WR,WR3,11,7,7.33,<NA>,6,9,...,17,300.4,17.7,270.76,0.33,WR,23070,6215db71-1e13-4c62-8f14-2005d33dee47,40041,3162959
7,Amon-Ra St. Brown,DET,WR,WR4,6,8,7.67,<NA>,7,8,...,17,265.5,15.6,261.15,-0.33,WR,19799,26ef0447-29b8-4dd1-9831-dcbf8132bb74,33500,2967690
8,James Cook III,BUF,RB,RB5,7,9,8.67,<NA>,8,9,...,17,285.7,16.8,253.93,-0.33,RB,22958,ce5de0e5-9f76-42ee-b93a-3ef88af9b0a7,34019,2975698
9,CeeDee Lamb,DAL,WR,WR5,14,10,11.33,<NA>,10,13,...,14,163.4,11.7,224.27,1.33,WR,19202,a72ea15b-5199-4101-a300-846e1c655add,32687,2865251


## Column order + dtypes

Columns must come back in the agreed `CHECKPOINT_COLUMNS` order. `tier` is
kept in the schema but is currently all-missing — the production half-PPR
pull sends `experts=show`, and FantasyPros drops the per-player `tier` field
from that response.

In [3]:
assert list(df.columns) == CHECKPOINT_COLUMNS, "column order drift"
df.dtypes

player_name                  str
team                         str
position                     str
pos_rank                     str
bye_wk                     Int64
rank_ecr                   int64
consensus_adp_half       float64
tier                       Int64
rank_min                   int64
rank_max                   int64
rank_avg                 float64
rank_std                 float64
rank_range                 int64
yahoo_adp_half           float64
rtsports_adp_half        float64
sleeper_adp_half         float64
espn_adp_ppr             float64
2025_games_played          Int64
2025_points_half         float64
2025_ppg_half            float64
projected_points_half    float64
adp_vs_ecr               float64
eligible_positions           str
player_id                  int64
sportsdata_id                str
player_yahoo_id              str
cbs_player_id                str
dtype: object

## Coverage / null checks

Enrichment is a LEFT JOIN, so partial coverage is expected. Missing values
must remain `NaN` / `<NA>` — never zero-filled.

In [4]:
report = checkpoint_coverage_report(df)
report

{'row_count': 365,
 'player_id_unique': True,
 'player_id_nulls': 0,
 'coverage': {'espn_adp_ppr': {'non_null': 242, 'pct': np.float64(66.3)},
  'yahoo_adp_half': {'non_null': 224, 'pct': np.float64(61.4)},
  'rtsports_adp_half': {'non_null': 298, 'pct': np.float64(81.6)},
  'sleeper_adp_half': {'non_null': 320, 'pct': np.float64(87.7)},
  'projected_points_half': {'non_null': 310, 'pct': np.float64(84.9)},
  '2025_games_played': {'non_null': 278, 'pct': np.float64(76.2)},
  '2025_points_half': {'non_null': 278, 'pct': np.float64(76.2)},
  '2025_ppg_half': {'non_null': 278, 'pct': np.float64(76.2)},
  'tier': {'non_null': 0, 'pct': np.float64(0.0)}},
 'min_projected_points_half': 0.0}

In [5]:
df.isna().sum().sort_values(ascending=False)

tier                     365
yahoo_adp_half           141
espn_adp_ppr             123
2025_points_half          87
2025_games_played         87
2025_ppg_half             87
rtsports_adp_half         67
bye_wk                    56
projected_points_half     55
sleeper_adp_half          45
player_name                0
rank_avg                   0
rank_max                   0
rank_min                   0
consensus_adp_half         0
pos_rank                   0
rank_ecr                   0
team                       0
position                   0
rank_range                 0
rank_std                   0
adp_vs_ecr                 0
eligible_positions         0
player_id                  0
sportsdata_id              0
player_yahoo_id            0
cbs_player_id              0
dtype: int64

## QA assertions

* output row count == the half-PPR consensus player universe
* `player_id` is unique (joins did not duplicate any player)
* enrichment `NaN` was not silently turned into `0`

In [6]:
with (RAW_DIR / "fantasypros_consensus_adp_2026_half.json").open() as f:
    spine_universe = len(json.load(f)["players"])

assert len(df) == spine_universe, (len(df), spine_universe)
assert df["player_id"].is_unique
assert df["player_id"].notna().all()
# zero-fill guard: an unmatched enrichment row must be NaN, not 0.
unmatched_2025 = df[df["2025_points_half"].isna()]
assert (unmatched_2025["2025_points_half"] != 0).all()

print(f"OK — {len(df)} players, player_id unique, no zero-fill")

OK — 365 players, player_id unique, no zero-fill


## `rank_ecr` vs `consensus_adp_half` — inspect side by side

Known ambiguity in the raw data:

In [7]:
print(RANK_ECR_VS_ADP_NOTE)
df[["player_name", "position", "rank_ecr", "consensus_adp_half", "rank_avg", "adp_vs_ecr"]].head(25)

The 2026 half-PPR cache is a type=ADP consensus-rankings pull. Its 'rank_ecr' field is the consensus ADP ordinal, not a separate expert ranking; 'rank_ave' is the averaged ADP value. consensus_adp_half is mapped from rank_ave (same source as rank_avg), so adp_vs_ecr here is (mean ADP slot - ADP ordinal), NOT market-ADP-vs-true-ECR. A real ECR comparison needs a separate type=ECR / /rankings pull that is not cached.


,player_name,position,rank_ecr,consensus_adp_half,rank_avg,adp_vs_ecr
0,Jahmyr Gibbs,RB,1,1.00,1.00,0.00
1,Bijan Robinson,RB,2,2.00,2.00,0.00
2,Ja'Marr Chase,WR,3,3.00,3.00,0.00
3,Puka Nacua,WR,4,4.67,4.67,0.67
4,Christian McCaffrey,RB,5,4.67,4.67,-0.33
5,Jonathan Taylor,RB,6,6.00,6.00,0.00
6,Jaxon Smith-Njigba,WR,7,7.33,7.33,0.33
7,Amon-Ra St. Brown,WR,8,7.67,7.67,-0.33
8,James Cook III,RB,9,8.67,8.67,-0.33
9,CeeDee Lamb,WR,10,11.33,11.33,1.33


## Export the Excel checkpoint

Single sheet, frozen header, autofilter, sensible widths. This is a review
artifact, **not** the final draft-day workbook.

In [8]:
out_path = write_checkpoint_excel(
    df, PROJECT_ROOT / "outputs" / "fantasy_draft_checkpoint_2026.xlsx"
)
print("wrote", out_path)

wrote /Users/braddotson/Desktop/Github/Fantasy_Football_Data_Pipeline/outputs/fantasy_draft_checkpoint_2026.xlsx
